# VeriCall — Execution-Verified Post-Training for Tool Calling
### Ground-Up Architecture with In-Context Candidate Schemas & Anti-Collapse DPO

This notebook implements the complete **VeriCall** post-training and evaluation pipeline on **Qwen2.5-7B-Instruct**:
1. **16 Enterprise REST Tool Schemas** (10 Training in Finance/Commerce/CRM, 6 Held-Out Unseen in Cloud Infra/Travel Logistics).
2. **In-Context Candidate Tool Schemas**: Every prompt injects 6 candidate JSON tool schemas (1 target + 5 domain decoys) under `Available Tools:`.
3. **ExecutionValidator**: 6-state error taxonomy (`passed`, `invalid_json`, `wrong_endpoint`, `missing_arg`, `hallucinated_arg`, `type_error`).
4. **Completion-Only SFT**: Masks prompt tokens so the model only trains on assistant completions (`{"name": "...", "arguments": {...}}`).
5. **Anti-Collapse DPO**: $\beta=0.1$, $\text{lr}=5\times 10^{-6}$, $\text{rpo\_alpha}=0.2$ NLL anchor on 1,000 near-miss preference pairs (40% hallucinated, 30% type, 20% missing, 10% wrong endpoint).
6. **Pre-Flight Sanity Gates**: 25-sample smoke assertion checks after each stage (`base > 20%`, `sft > base`, `dpo >= sft`).
7. **Multi-Model Evaluation**: Evaluates Base, Constrained Decoding, SFT, and SFT+DPO across $M=500$ held-out requests with 10,000 bootstrap CIs and paired McNemar tests.


### Stage 1: Environment Setup & Dependencies


In [ ]:
# Install dependencies with unsloth and standard HF stack
import os
import sys

print("[*] Installing and verifying dependencies...")
!pip install -q "fastapi>=0.100.0" "pydantic>=2.0.0" "httpx>=0.24.0" "statsmodels>=0.14.0" "scipy>=1.10.0" "tabulate>=0.9.0"

# Attempt unsloth fast install
try:
    import unsloth
    print("[✓] Unsloth is already installed.")
    USE_UNSLOTH = True
except ImportError:
    print("[*] Installing unsloth for 2x faster 4-bit LoRA training...")
    !pip install -q "transformers>=4.40.0" "peft>=0.9.0" "accelerate>=0.28.0" "bitsandbytes>=0.43.0"
    !pip install -q "trl<0.12.0" || pip install -q trl
    try:
        import unsloth
        USE_UNSLOTH = True
        print("[✓] Unsloth installed successfully.")
    except Exception as e:
        print(f"[!] Unsloth install fallback: using Hugging Face PEFT + TRL ({e})")
        USE_UNSLOTH = False

# Verify and set working directory to repository root (Supports Kaggle, Colab, Local)
from pathlib import Path

if os.path.exists("/kaggle/working"):
    repo_dir = "/kaggle/working/vericall"
    if not os.path.exists(repo_dir):
        print(f"[*] Cloning repository to {repo_dir}...")
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    else:
        print(f"[*] Pulling latest changes into {repo_dir}...")
        os.system(f"cd {repo_dir} && (git pull origin master || git pull origin main)")
    os.chdir(repo_dir)
elif os.path.exists("/content"):
    repo_dir = "/content/vericall"
    if not os.path.exists(repo_dir):
        print(f"[*] Cloning repository to {repo_dir}...")
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    else:
        print(f"[*] Pulling latest changes into {repo_dir}...")
        os.system(f"cd {repo_dir} && (git pull origin master || git pull origin main)")
    os.chdir(repo_dir)
else:
    curr = Path.cwd()
    if (curr / "pyproject.toml").exists():
        print(f"[✓] Already at repository root: {curr}")
    elif (curr.parent / "pyproject.toml").exists():
        os.chdir(str(curr.parent))
    else:
        print(f"[!] Current directory: {curr}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"[✓] Environment ready. Working directory: {os.getcwd()}")
print(f"[✓] Dependency setup complete. Unsloth engine active: {USE_UNSLOTH}")


### Stage 2: In-Process FastAPI Mock Sandbox & ExecutionValidator (16 Endpoints)


In [ ]:
from vericall.sandbox.validator import ExecutionValidator
from vericall.sandbox.app import app, ENDPOINT_REGISTRY, get_tool_json_schema

validator = ExecutionValidator()
print(f"[✓] Initialized ExecutionValidator with {len(validator.registry)} endpoints across 5 domains.")

# Smoke test all 16 endpoints
test_endpoints = [
    ("issue_refund", {"transaction_id": "txn_12345678", "amount": 49.99, "currency": "USD", "reason": "CUSTOMER_RETURN", "metadata": {"rep": "rep_1"}}),
    ("create_invoice", {"client_id": "cli_1001", "issue_date": "2026-11-01", "due_date": "2026-11-15", "line_items": [{"description": "Dev", "unit_price": 100.0, "quantity": 1}], "currency": "USD"}),
    ("transfer_funds", {"source_account": "acc_100001", "destination_account": "acc_100002", "amount": 250.0, "transfer_method": "ACH", "memo": "Payment"}),
    ("audit_ledger_entries", {"ledger_id": "ledg_100001", "start_date": "2026-10-01", "end_date": "2026-10-31", "entry_type": "CREDIT", "filter_flags": {"reconciled_only": True}}),
    ("query_inventory", {"sku_list": ["SKU-PROD01"], "warehouse_id": "wh_us-01", "include_out_of_stock": False, "fulfillment_channel": "WAREHOUSE_DIRECT"}),
    ("apply_discount_code", {"cart_id": "cart_100001", "promo_code": "SAVE20", "channel": "ONLINE_WEB", "customer_loyalty_pin": 1234}),
    ("update_subscription", {"subscription_id": "sub_10000001", "plan_tier": "PRO_MONTHLY", "auto_renew": True, "addon_features": ["logs"], "effective_date": "2026-12-01"}),
    ("create_support_ticket", {"customer_id": "cust_10001", "priority": "HIGH", "department": "TECHNICAL", "subject": "Platform latency issue", "ticket_context": {"origin": "web"}}),
    ("update_customer_tier", {"customer_id": "cust_10001", "target_tier": "GOLD", "assigned_rep_id": "rep_1001", "override_reason": "High spending account", "benefits": ["waived_fees"]}),
    ("schedule_webhook", {"destination_url": "https://api.partner.com/events", "subscribed_events": ["order.created"], "retry_count": 3, "signing_secret": "whsec_12345678", "headers": {"X-App": "test"}}),
    ("provision_compute_node", {"instance_tier": "c6i.xlarge", "cloud_region": "us-east-1", "cluster_count": 2, "environment_tag": "PRODUCTION", "resource_tags": {"team": "infra"}}),
    ("scale_kubernetes_cluster", {"cluster_name": "k8s-prod-101", "node_pool": "worker-pool", "desired_replicas": 5, "scaling_strategy": "SURGE_FIRST", "autoscale_config": {"min_nodes": 2, "max_nodes": 20}}),
    ("rotate_iam_credentials", {"service_account_id": "sa-data1001", "credential_type": "OAUTH2_KEY", "grace_period_hours": 48, "notify_emails": ["ops@company.com"], "force_revocation": False}),
    ("search_multi_leg_flights", {"itinerary_segments": [{"departure_airport": "JFK", "arrival_airport": "LHR", "flight_date": "2026-11-10"}], "cabin_class": "BUSINESS", "passenger_count": 1, "flexible_dates": False, "preferred_alliances": ["STAR_ALLIANCE"]}),
    ("reserve_corporate_hotel", {"property_code": "HTL-NY-1001", "check_in_date": "2026-12-01", "check_out_date": "2026-12-05", "room_category": "EXECUTIVE_SUITE", "corporate_billing_code": "CORP-10001", "guest_manifest": [{"first_name": "Jane", "last_name": "Doe"}]}),
    ("cancel_itinerary_segment", {"booking_reference": "PNR-ABC123", "segment_id": "seg_01", "cancellation_reason": "CLIENT_RESCHEDULE", "request_credit_voucher": True, "refund_override_notes": "Authorized"}),
]

for fn, args in test_endpoints:
    res = validator.validate_call({"name": fn, "arguments": args}, expected_func=fn)
    assert res.is_success, f"Endpoint {fn} failed: {res.error_message}"

print("[✓] All 16 sandbox endpoints verified functional with HTTP 200 execution.")


### Stage 3: Load Model & Tokenizer (Dual Engine: Unsloth / PEFT 4-Bit)


In [ ]:
import torch
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-7B-Instruct"
max_seq_length = 1536

print(f"[*] Loading {model_id} in 4-bit...")

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    print("[✓] Loaded model via unsloth FastLanguageModel.")
else:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    print("[✓] Loaded model via standard HuggingFace BitsAndBytes.")

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[✓] Tokenizer configured (vocab_size={len(tokenizer)}). Ready for evaluation & training.")


### Stage 4: Base Model & Constrained Decoding Evaluation (with 25-Sample Pre-Flight Gate)


In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
from vericall.eval.evaluate import evaluate_predictions, apply_constrained_decoding

# Load M=500 unseen API test set (each item has prompt with 6 candidate schemas)
unseen_path = Path("data/eval_unseen_apis.jsonl")
test_records = [json.loads(line) for line in open(unseen_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(test_records)} frozen unseen API evaluation requests.")

# Check candidate tools in prompt
assert "Available Tools:" in test_records[0]["prompt"], "Prompt missing candidate tool schemas!"
print(f"[✓] Verified in-prompt candidate schemas present. Target: {test_records[0]['target_function']}")

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()

device = next(model.parameters()).device

# --- 25-SAMPLE PRE-FLIGHT GATE ---
print("\n[*] Running 25-sample Base Model Pre-Flight Smoke Check...")
smoke_preds = []
for item in test_records[:25]:
    prompt = item["prompt"]
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=160, use_cache=True, temperature=0.0, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    smoke_preds.append(resp)

smoke_res = evaluate_predictions(test_records[:25], smoke_preds, "base_smoke")
base_smoke_acc = smoke_res["accuracy_pct"]
print(f"[Base Smoke Check] Passed: {smoke_res['passed_requests']}/25 ({base_smoke_acc}%)")
print(f"                   Errors: {smoke_res['error_taxonomy']}")

# PRE-FLIGHT ASSERTION: With candidate schemas in prompt, Base model must achieve > 20%
assert base_smoke_acc > 20.0, f"PRE-FLIGHT FAILURE: Base accuracy ({base_smoke_acc}%) <= 20.0%! Check prompt schemas."
print(f"[✓] PRE-FLIGHT GATE PASSED: Base model achieved {base_smoke_acc}% (> 20.0% threshold).")

# --- FULL 500-SAMPLE EVALUATION ---
print("\n[*] Generating predictions on full M=500 unseen API test set with Base Model...")
base_predictions = smoke_preds.copy()

for item in tqdm(test_records[25:]):
    prompt = item["prompt"]
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=160, use_cache=True, temperature=0.0, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    base_predictions.append(resp)

base_results = evaluate_predictions(test_records, base_predictions, "base_model")
print(f"\n[Base Model] Unseen Accuracy: {base_results['accuracy_pct']}% (Passed: {base_results['passed_requests']}/{base_results['total_requests']})")
print(f"             95% Bootstrap CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"             Errors: {base_results['error_taxonomy']}")

# Evaluate Base + Constrained Decoding
print("\n[*] Evaluating Base Model + Constrained Schema Decoding...")
constrained_preds = []
for item, pred in zip(test_records, base_predictions):
    cand_tools = item["candidate_tools"]
    try:
        parsed = json.loads(pred) if isinstance(pred, str) else pred
    except Exception:
        parsed = {"name": cand_tools[0]["name"], "arguments": {}}
    cleaned = apply_constrained_decoding(parsed, cand_tools)
    constrained_preds.append(cleaned)

constrained_results = evaluate_predictions(test_records, constrained_preds, "base_constrained")
print(f"[Constrained Decoding] Accuracy: {constrained_results['accuracy_pct']}% (Passed: {constrained_results['passed_requests']}/{constrained_results['total_requests']})")
print(f"                      95% CI: [{constrained_results['bootstrap_95_ci'][0]}%, {constrained_results['bootstrap_95_ci'][1]}%]")
print(f"                      Errors: {constrained_results['error_taxonomy']}")


### Stage 5: QLoRA Supervised Fine-Tuning (SFT) with Completion-Only Loss Masking


In [ ]:
from trl import SFTTrainer
from datasets import Dataset
import inspect

# Configure LoRA targeting all linear projections
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
else:
    from peft import LoraConfig, get_peft_model
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)

# Load 2,000 training records
train_file = Path("data/train_sft.jsonl")
train_records = [json.loads(line) for line in open(train_file, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(train_records)} SFT training records across 10 Finance/Commerce/CRM endpoints.")

train_dataset = Dataset.from_list([{"text": r["text"]} for r in train_records])

# Completion-only loss masking on assistant turn (Resilient fallback for all TRL versions)
response_template = "<|im_start|>assistant\n"

def get_completion_collator(response_template, tokenizer):
    try:
        from trl import DataCollatorForCompletionOnlyLM
        print("[✓] Loaded DataCollatorForCompletionOnlyLM from trl.")
        return DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)
    except (ImportError, AttributeError):
        print("[*] Using resilient custom DataCollatorForCompletionOnlyLM fallback...")
        class SafeCompletionCollator:
            def __init__(self, resp_tpl, tok):
                self.tok = tok
                self.resp_tpl = resp_tpl
                self.resp_ids = tok.encode(resp_tpl, add_special_tokens=False)

            def __call__(self, examples):
                texts = [e["text"] for e in examples]
                batch = self.tok(texts, padding=True, truncation=True, max_length=1536, return_tensors="pt")
                labels = batch["input_ids"].clone()
                n = len(self.resp_ids)
                for i in range(labels.shape[0]):
                    ids = batch["input_ids"][i].tolist()
                    found = -1
                    for j in range(len(ids) - n + 1):
                        if ids[j:j+n] == self.resp_ids:
                            found = j + n
                            break
                    if found != -1:
                        labels[i, :found] = -100
                    else:
                        labels[i, :] = -100
                batch["labels"] = labels
                return batch
        return SafeCompletionCollator(response_template, tokenizer)

collator = get_completion_collator(response_template, tokenizer)

# Ensure trainable parameters in float32 for AdamW stability
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

try:
    from trl import SFTConfig
    sft_args = SFTConfig(
        output_dir="checkpoints/runs_sft",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        optim="adamw_torch",
        warmup_steps=10,
        lr_scheduler_type="cosine",
        logging_steps=20,
        seed=42,
        fp16=False,
        bf16=False,
        report_to="none",
        max_length=max_seq_length,
        dataset_text_field="text",
    )
except ImportError:
    from transformers import TrainingArguments
    sft_args = TrainingArguments(
        output_dir="checkpoints/runs_sft",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        optim="adamw_torch",
        warmup_steps=10,
        lr_scheduler_type="cosine",
        logging_steps=20,
        seed=42,
        fp16=False,
        bf16=False,
        report_to="none",
    )

sft_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "data_collator": collator,
    "args": sft_args,
}
sig = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in sig:
    sft_kwargs["processing_class"] = tokenizer
else:
    sft_kwargs["tokenizer"] = tokenizer

sft_trainer = SFTTrainer(**sft_kwargs)

print("[*] Starting QLoRA SFT training with completion-only loss masking...")
sft_trainer.train()

model.save_pretrained("checkpoints/sft_adapter")
tokenizer.save_pretrained("checkpoints/sft_adapter")
print("[✓] SFT training complete. Adapters saved to checkpoints/sft_adapter.")


### Stage 6: Evaluate SFT Model on M=500 Unseen APIs (with Pre-Flight Gate)


In [ ]:
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()

device = next(model.parameters()).device

# --- 25-SAMPLE PRE-FLIGHT GATE ---
print("\n[*] Running 25-sample SFT Pre-Flight Smoke Check...")
smoke_sft_preds = []
for item in test_records[:25]:
    prompt = item["prompt"]
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=160, use_cache=True, temperature=0.0, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    smoke_sft_preds.append(resp)

smoke_sft_res = evaluate_predictions(test_records[:25], smoke_sft_preds, "sft_smoke")
sft_smoke_acc = smoke_sft_res["accuracy_pct"]
print(f"[SFT Smoke Check] Passed: {smoke_sft_res['passed_requests']}/25 ({sft_smoke_acc}%) vs Base ({base_smoke_acc}%)")
print(f"                  Errors: {smoke_sft_res['error_taxonomy']}")

# PRE-FLIGHT ASSERTION: SFT must show gain over Base
assert sft_smoke_acc >= base_smoke_acc, f"PRE-FLIGHT FAILURE: SFT ({sft_smoke_acc}%) did not improve over Base ({base_smoke_acc}%)!"
assert sft_smoke_acc > 30.0, f"PRE-FLIGHT FAILURE: SFT accuracy ({sft_smoke_acc}%) <= 30%!"
print(f"[✓] PRE-FLIGHT GATE PASSED: SFT accuracy ({sft_smoke_acc}%) exceeds Base ({base_smoke_acc}%).")

# --- FULL 500-SAMPLE EVALUATION ---
print("\n[*] Generating predictions on full M=500 unseen API test set with SFT Model...")
sft_predictions = smoke_sft_preds.copy()

for item in tqdm(test_records[25:]):
    prompt = item["prompt"]
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=160, use_cache=True, temperature=0.0, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    sft_predictions.append(resp)

sft_results = evaluate_predictions(test_records, sft_predictions, "sft_only")
print(f"\n[SFT Model] Unseen Accuracy: {sft_results['accuracy_pct']}% (Passed: {sft_results['passed_requests']}/{sft_results['total_requests']})")
print(f"            95% Bootstrap CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"            Errors: {sft_results['error_taxonomy']}")


### Stage 7: Execution-Grounded Near-Miss DPO Mining


In [ ]:
from vericall.data.mine_dpo import mine_dpo_pairs

dpo_pairs = mine_dpo_pairs(num_pairs=1000)
print(f"[✓] Successfully loaded/mined {len(dpo_pairs)} execution-verified preference pairs.")
print("Sample Preference Pair:")
print("  Prompt (first 120 chars):", dpo_pairs[0]["prompt"][:120].replace('\n', ' '))
print("  Chosen:                  ", dpo_pairs[0]["chosen"])
print("  Rejected:                ", dpo_pairs[0]["rejected"], f"({dpo_pairs[0]['rejected_status']})")


### Stage 8: QLoRA Direct Preference Optimization (DPO) with Anti-Collapse Safeguards


In [ ]:
import os
import inspect
import torch
from datasets import Dataset
from trl import DPOTrainer, DPOConfig

# Anti-collapse hyperparameters:
# beta = 0.1
# learning_rate = 5e-6 (conservative rate avoids policy collapse)
# rpo_alpha = 0.2 (adds NLL regularization anchor on chosen completion)
# max_length = 1536, max_prompt_length = 1280

if USE_UNSLOTH:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()

cfg_params = inspect.signature(DPOConfig.__init__).parameters
dpo_cfg_kwargs = {
    "output_dir": "checkpoints/runs_dpo",
    "beta": 0.1,
    "learning_rate": 5e-6,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs": 1,
    "optim": "adamw_torch",
    "warmup_steps": 10,
    "lr_scheduler_type": "cosine",
    "logging_steps": 10,
    "seed": 42,
    "fp16": False,
    "bf16": False,
    "report_to": "none",
    "max_length": 1536,
    "max_prompt_length": 1280,
}

if "rpo_alpha" in cfg_params:
    dpo_cfg_kwargs["rpo_alpha"] = 0.2
    print("[✓] Enabled RPO NLL anchor (rpo_alpha=0.2) to mathematically prevent policy collapse.")

dpo_config = DPOConfig(**dpo_cfg_kwargs)

dpo_train_dataset = Dataset.from_list([
    {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
    for p in dpo_pairs
])

# Cast trainable parameters to float32
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

dpo_kwargs = {
    "model": model,
    "ref_model": None,
    "args": dpo_config,
    "train_dataset": dpo_train_dataset,
}
sig_dpo = inspect.signature(DPOTrainer.__init__).parameters
if "processing_class" in sig_dpo:
    dpo_kwargs["processing_class"] = tokenizer
else:
    dpo_kwargs["tokenizer"] = tokenizer

dpo_trainer = DPOTrainer(**dpo_kwargs)

print("[*] Starting QLoRA DPO alignment with anti-collapse safeguards...")
dpo_trainer.train()

model.save_pretrained("checkpoints/vericall-dpo-lora")
tokenizer.save_pretrained("checkpoints/vericall-dpo-lora")
print("[✓] DPO training complete. Adapters saved to checkpoints/vericall-dpo-lora.")


### Stage 9: Final Evaluation & Statistical Rigor (Bootstrap CI & McNemar Tests)


In [ ]:
from vericall.eval.stats import compute_mcnemar_test

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()

device = next(model.parameters()).device

# --- 25-SAMPLE PRE-FLIGHT GATE ---
print("\n[*] Running 25-sample DPO Pre-Flight Smoke Check...")
smoke_dpo_preds = []
for item in test_records[:25]:
    prompt = item["prompt"]
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=160, use_cache=True, temperature=0.0, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    smoke_dpo_preds.append(resp)

smoke_dpo_res = evaluate_predictions(test_records[:25], smoke_dpo_preds, "dpo_smoke")
dpo_smoke_acc = smoke_dpo_res["accuracy_pct"]
print(f"[DPO Smoke Check] Passed: {smoke_dpo_res['passed_requests']}/25 ({dpo_smoke_acc}%) vs SFT ({sft_smoke_acc}%)")
print(f"                  Errors: {smoke_dpo_res['error_taxonomy']}")

# PRE-FLIGHT ASSERTION: DPO must not collapse (in previous run it scored 0.0%)
assert dpo_smoke_acc >= (sft_smoke_acc - 4.0), f"PRE-FLIGHT FAILURE: DPO ({dpo_smoke_acc}%) collapsed below SFT ({sft_smoke_acc}%)!"
assert dpo_smoke_acc > 30.0, f"PRE-FLIGHT FAILURE: DPO collapsed to {dpo_smoke_acc}%!"
print(f"[✓] PRE-FLIGHT GATE PASSED: DPO holds or outperforms SFT ({dpo_smoke_acc}% vs SFT {sft_smoke_acc}%).")

# --- FULL 500-SAMPLE EVALUATION ---
print("\n[*] Generating final predictions with VeriCall (SFT+DPO) on full M=500 unseen APIs...")
dpo_predictions = smoke_dpo_preds.copy()

for item in tqdm(test_records[25:]):
    prompt = item["prompt"]
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=160, use_cache=True, temperature=0.0, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    dpo_predictions.append(resp)

dpo_results = evaluate_predictions(test_records, dpo_predictions, "sft_dpo_vericall")

# Compute paired McNemar tests
stat_base, p_base = compute_mcnemar_test(base_results["binary_scores"], dpo_results["binary_scores"])
stat_sft, p_sft = compute_mcnemar_test(sft_results["binary_scores"], dpo_results["binary_scores"])

print(f"\n=======================================================")
print(f"FINAL VERICALL BENCHMARK RESULTS (M=500 Unseen APIs)")
print(f"=======================================================")
print(f"[Base Model]     Accuracy: {base_results['accuracy_pct']}%  95% CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"[Base + Constr]  Accuracy: {constrained_results['accuracy_pct']}%  95% CI: [{constrained_results['bootstrap_95_ci'][0]}%, {constrained_results['bootstrap_95_ci'][1]}%]")
print(f"[SFT-Only]       Accuracy: {sft_results['accuracy_pct']}%  95% CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"[VeriCall (DPO)] Accuracy: {dpo_results['accuracy_pct']}%  95% CI: [{dpo_results['bootstrap_95_ci'][0]}%, {dpo_results['bootstrap_95_ci'][1]}%]")
print(f"\nMcNemar Test vs Base:    chi2 = {stat_base:.4f}, p = {p_base:.4e} (Significant: {p_base < 0.05})")
print(f"McNemar Test vs SFT:     chi2 = {stat_sft:.4f}, p = {p_sft:.4e} (Significant: {p_sft < 0.05})")
print(f"=======================================================")


### Stage 10: Save Evaluation Summary & Render Results


In [ ]:
from pathlib import Path
import json
import os
import shutil

summary_data = {
    "metadata": {
        "model_base": model_id,
        "eval_set": "eval_unseen_apis.jsonl",
        "eval_count": len(test_records),
        "mode": "full_benchmark_v2",
        "device": str(device),
        "num_dpo_pairs": len(dpo_pairs),
    },
    "configs": {
        "base_model": base_results,
        "base_constrained": constrained_results,
        "sft_only": sft_results,
        "sft_dpo_vericall": dpo_results,
    },
    "mcnemar_tests": {
        "vericall_vs_base": {"statistic": stat_base, "p_value": p_base},
        "vericall_vs_sft": {"statistic": stat_sft, "p_value": p_sft},
    }
}

Path("results").mkdir(exist_ok=True)
with open("results/eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

print("[✓] Saved evaluation summary to results/eval_summary.json")

# Copy to /kaggle/working for direct download
if os.path.exists("/kaggle/working"):
    shutil.copy2("results/eval_summary.json", "/kaggle/working/eval_summary.json")
    print("[✓] Copied to /kaggle/working/eval_summary.json for direct artifact download")

!python scripts/render_results.py
